In [25]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "requests", "pandas"], capture_output=True)

import requests, pandas as pd

API_KEY = "f0970d98c4f6ff4045c452e5e5743fe5"
BASE    = "https://manifesto-project.wzb.eu/api/v1"

print("Downloading main dataset...")
r = requests.get(
    f"{BASE}/get_core",
    params={"api_key": API_KEY, "key": "MPDS2023a"},
    timeout=60
)
print(f"Status: {r.status_code}")

data = r.json()

# First row is headers, rest is data
full = pd.DataFrame(data[1:], columns=data[0])
print(f"Dataset loaded: {full.shape[0]} rows, {full.shape[1]} columns")

# Filter to Germany 2021
ger21 = full[
    (full["countryname"] == "Germany") &
    (full["edate"].astype(str).str.contains("2021"))
].copy()

print(f"German 2021 parties found: {len(ger21)}\n")

# ── TABLE A4: RILE scores ─────────────────────────────────────────────────────
print("=" * 65)
print("TABLE A4: RILE SCORES")
print("=" * 65)
print(f"{'Party':<50} {'RILE':>8}")
print("-" * 65)
for _, row in ger21[["partyname", "rile"]].iterrows():
    print(f"{str(row['partyname']):<50} {str(row['rile']):>8}")

# ── TABLE A5: topic shares ────────────────────────────────────────────────────
topic_map = {
    "per202": "Support of democracy",
    "per303": "Governmental/administrative efficiency",
    "per401": "Free market economy and capitalism",
    "per411": "Technology, infrastructure, modernisation",
    "per504": "Welfare state expansion",
    "per601": "National way of life: nation and history",
    "per608": "National way of life: restrict immigration",
    "per302": "Importance of personal freedom, civil rights",
    "per416": "Improve/expand educational provision",
    "per501": "Environmental protection",
    "per107": "Peace, anti-militarism, internationalism",
    "per503": "Provision of cultural/leisure facilities",
}

available = {k: v for k, v in topic_map.items() if k in ger21.columns}
missing   = [k for k in topic_map if k not in ger21.columns]
if missing:
    print(f"Note: columns not in dataset: {missing}")

topic_df = ger21[["partyname"] + list(available.keys())].copy()
topic_df = topic_df.set_index("partyname")
topic_df.columns = [available[c] for c in topic_df.columns]
topic_df = topic_df.apply(pd.to_numeric, errors="coerce").round(1)

print("\n" + "=" * 65)
print("TABLE A5: TOPIC SHARES (% of quasi-sentences per party)")
print("=" * 65)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 50)
print(topic_df.T.to_string())

print("\n" + "=" * 65)
print("DONE")
print("=" * 65)

Status: 200
Dataset loaded: 5089 rows, 175 columns
German 2021 parties found: 7

TABLE A4: RILE SCORES
Party                                                  RILE
-----------------------------------------------------------------
Alliance‘90/Greens                                  -21.038
The Left                                            -36.167
Social Democratic Party of Germany                  -24.673
Free Democratic Party                                 0.266
Christian Democratic Union/Christian Social Union     3.495
South Schleswig Voters’ Union                        -22.45
Alternative for Germany                              26.048

TABLE A5: TOPIC SHARES (% of quasi-sentences per party)
partyname                                     Alliance‘90/Greens  The Left  Social Democratic Party of Germany  Free Democratic Party  Christian Democratic Union/Christian Social Union  South Schleswig Voters’ Union  Alternative for Germany
Support of democracy                                 